# Data Cleanin Project with Python

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('marketing_campaign_data_messy.csv')

print (f"loaded datasets: {df.shape[0]} rows, {df.shape[1]} columns")

loaded datasets: 2020 rows, 12 columns


In [ ]:
df

,Campaign_ID,Campaign_Name,Start_Date,End_Date,Channel,Impressions,Clicks,Spend,Conversions,Active,Clicks,Campaign_Tag
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24 00:00:00,2023-12-13,TikTok,16795,197,$102.82,20.0,Y,NaN,TI
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06 00:00:00,2023-05-12,Facebook,1860,30,24.33,1.0,0,NaN,FA
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13 00:00:00,2023-12-20,Email,77820,843,1323.39,51.0,No,NaN,EM
3,CMP-00004,Q1_BlackFriday_CMP-00004,2023-10-30,2023-11-03,TikTok,55886,2019,2180.38,135.0,True,NaN,TI
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22 00:00:00,2023-04-23,Facebook,7265,169,252.44,30.0,Yes,NaN,FA
...,...,...,...,...,...,...,...,...,...,...,...,...
2015,CMP-00400,Q3_Summer_CMP-00400,2023-10-31 00:00:00,2023-11-13,TikTok,30592,586,$503.95,77.0,1,NaN,TI
2016,CMP-01255,Q4_Summer_CMP-01255,2023-09-01 00:00:00,2023-09-26,Google Ads,20097,897,1641.0,162.0,0,NaN,GO
2017,CMP-01050,Q2_Launch_CMP-01050,2023-02-09 00:00:00,2023-02-21,Instagram,33254,1117,883.82,214.0,0,NaN,IN
2018,CMP-01118,Q4_Winter_CMP-01118,2023-03-30 00:00:00,2023-04-27,Facebook,68728,2960,4198.5,591.0,Yes,NaN,FA


Header Cleaning

In [ ]:
print(df.columns.tolist())

df.columns = df.columns.str.strip().str.title().str.replace('_', ' ')

print('FIX APPLIED')
print(df.columns.tolist())

[' Campaign_ID ', 'Campaign_Name', 'Start_Date', 'End_Date', 'Channel', 'Impressions', 'Clicks ', 'Spend', 'Conversions', 'Active', 'Clicks', 'Campaign_Tag']
FIX APPLIED
['Campaign Id', 'Campaign Name', 'Start Date', 'End Date', 'Channel', 'Impressions', 'Clicks', 'Spend', 'Conversions', 'Active', 'Clicks', 'Campaign Tag']


Type Convertion & Currency Cleanig

In [ ]:
dirty_spend_mask = df['Spend'].astype(str).str.contains(r'\$')
print(df.loc[dirty_spend_mask, ['Campaign Id','Spend']].head(3))

df['Spend'] = df['Spend'].str.replace(r'[^\d.-]', '', regex= True)
df['Spend'] = pd.to_numeric(df['Spend'])

print('FIX APPLIED')
print(df.loc[dirty_spend_mask, ['Campaign Id','Spend']].head(3))


   Campaign Id     Spend
0    CMP-00001   $102.82
21   CMP-00022   $2428.4
22   CMP-00023  $4726.22
FIX APPLIED
   Campaign Id    Spend
0    CMP-00001   102.82
21   CMP-00022  2428.40
22   CMP-00023  4726.22


Catagorical Typos Fuzzy Logic

In [ ]:
print(df['Channel'].unique())

cleanup_map = {
    'Facebok': 'Facebook',
    'Insta_gram': 'Instagram',
    'Gogle': 'Google Ads',
    'Tik_Tok': 'TikTok',
    'E-mail': 'Email',
    'N/A': np.nan
}
df = df.replace({'Channel': cleanup_map})

print('FIX APPLIED')
print(df['Channel'].unique())

['TikTok' 'Facebook' 'Email' 'Instagram' 'Google Ads' nan]
FIX APPLIED
['TikTok' 'Facebook' 'Email' 'Instagram' 'Google Ads' nan]


Handling Mixed Boolens

In [ ]:
print(df['Active'].unique().tolist())

bool_map = {
    'Yes': True,
    'Y': True,
    '1': True,
    'No': False,
    '0': False,
    0: False,
    1: True,

}
df['Active'] = df['Active'].map(bool_map).fillna(False).astype(bool)

print('FIX APPLIED')
print(df['Active'].unique().tolist())

[True, False]
FIX APPLIED
[True, False]


Data Parsing

In [ ]:
print(df['Start Date'].dtype)

df['Start Date'] = pd.to_datetime(df['Start Date'], errors='coerce')
df['End Date'] = pd.to_datetime(df['End Date'], errors='coerce')

print('FIX APPLIED')
print(df['Start Date'].dtype)


object
FIX APPLIED
datetime64[ns]


Logical Intigrity (Clicks vs Impressions)

In [ ]:
df = df.loc[:, ~df.columns.duplicated()]

In [ ]:
impossible_mask = df['Clicks'] > df['Impressions']
print(df.loc[impossible_mask, ['Campaign Id', 'Impressions', 'Clicks']].head(3))

Empty DataFrame
Columns: [Campaign Id, Impressions, Clicks]
Index: []


Logical Integrity Time Travel

In [ ]:
time_travel_mask = df['End Date'] < df['Start Date']
print(df.loc[time_travel_mask, ['Campaign Id', 'Start Date', 'End Date']].head(3))

df.loc[time_travel_mask, 'Start Date'] = df.loc[time_travel_mask, 'End Date'] + pd.Timedelta(days=30)

print('FIX APPLIED')
print(df.loc[time_travel_mask, ['Campaign Id', 'Start Date', 'End Date']].head(3))

   Campaign Id Start Date   End Date
23   CMP-00024 2023-05-06 2023-05-01
54   CMP-00055 2023-09-01 2023-08-27
71   CMP-00072 2023-02-01 2023-01-27
FIX APPLIED
   Campaign Id Start Date   End Date
23   CMP-00024 2023-05-31 2023-05-01
54   CMP-00055 2023-09-26 2023-08-27
71   CMP-00072 2023-02-26 2023-01-27


Handling Outliers (Winsoirizing)

In [ ]:
Q1 = df['Spend'].quantile(0.25)
Q3 = df['Spend'].quantile(0.75)

IQR = Q3 -Q3
upper_limit = Q1 + (3 * IQR)
lower_limit = Q1 - (3 * IQR)

outlier_mask = (df['Spend'] > upper_limit)
print(df.loc[outlier_mask, ['Campaign Id', 'Spend']].head(3))

print('FIX APPLIED')
df.loc[outlier_mask, 'Spend'] = upper_limit
print(df.loc[outlier_mask, ['Campaign Id', 'Spend']].head(3))

  Campaign Id    Spend
2   CMP-00003  1323.39
3   CMP-00004  2180.38
5   CMP-00006  2697.03
FIX APPLIED
  Campaign Id     Spend
2   CMP-00003  649.9975
3   CMP-00004  649.9975
5   CMP-00006  649.9975


String Pasing Future Extraction

In [ ]:
print (df['Campaign Name'].head(3))

df['Season'] = df['Campaign Name'].str.extract(r'Q\d_([^_]+)_')

print('FIX APPLIED')
print(df[['Campaign Name','Season']].head(3))

0    Q4_Summer_CMP-00001
1    Q1_Launch_CMP-00002
2    Q3_Winter_CMP-00003
Name: Campaign Name, dtype: object
FIX APPLIED
         Campaign Name  Season
0  Q4_Summer_CMP-00001  Summer
1  Q1_Launch_CMP-00002  Launch
2  Q3_Winter_CMP-00003  Winter


Hanling Nulls

In [ ]:
df[['Conversions', 'Channel', 'Start Date']].isnull().sum()

values = {
    'Conversions': df['Conversions'].mean(),
    'Channel': 'Unknown',
    'Start Date': df['Start Date'].ffill()
}

df.fillna(value=values, inplace=True)

print(df['Conversions'].isnull().sum())

0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2020 entries, 0 to 2019
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Campaign Id    2020 non-null   object        
 1   Campaign Name  2020 non-null   object        
 2   Start Date     2020 non-null   datetime64[ns]
 3   End Date       2020 non-null   datetime64[ns]
 4   Channel        2020 non-null   object        
 5   Impressions    2020 non-null   int64         
 6   Clicks         2020 non-null   int64         
 7   Spend          2020 non-null   float64       
 8   Conversions    2020 non-null   float64       
 9   Active         2020 non-null   bool          
 10  Campaign Tag   2020 non-null   object        
 11  Season         2020 non-null   object        
dtypes: bool(1), datetime64[ns](2), float64(2), int64(2), object(5)
memory usage: 175.7+ KB


In [ ]:
df.to_csv('cleaned_marketing_campaign_data.csv', index=False)
#